# TUS-GAN v4.1 (Cell C) — Kaggle Dual GPU Training Pipeline
**Architecture:** Convolutional Conditional GAN (Cell C)  
**Enforcement:** Deterministic Hard Logit Masking (Pre-softmax clamping to -1e9 for child work)  
**Hardware:** Kaggle Dual Tesla T4 GPUs (2x 16GB VRAM, DDP via `torchrun`)  
**Target Exhibit:** Table 6 (2x2 Factorial Matrix: Cell C Isolation) & Figure 8 (Dead-Gradient Autopsy)

---

### Step 1: Verify Hardware & GPU Environment
Confirm that Kaggle Accelerator is set to **GPU T4 x2**.

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
gpu_count = torch.cuda.device_count()
print(f"Detected GPUs  : {gpu_count}")
for i in range(gpu_count):
    print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB VRAM)")
assert gpu_count >= 1, "CUDA GPU not detected! Please enable GPU in notebook settings."

### Step 2: Verify Working Directory & Dataset
Verify that all required scripts and `tusgan_encode.npz` are present.

In [ ]:
!ls -lh

import os
assert os.path.exists("tusgan_encode.npz"), "tusgan_encode.npz not found in current directory!"
print("✓ Dataset verified: tusgan_encode.npz is present.")

### Step 3: Run Pre-Flight Smoke Test
Validates model shapes, hard masking logic, WGAN-GP autograd backward pass, and mathematically proves the **Dead-Gradient** characteristic of Cell C.

In [ ]:
!python smoke_test.py

### Step 4: Launch Full 250-Epoch Training on Dual GPU via `torchrun`
Uses PyTorch Distributed Data Parallel (DDP) across both Tesla T4 GPUs (`--nproc_per_node=2`), saving checkpoints every 10 epochs and logging gradient diagnostics at epochs 10, 50, 100, and 250.

In [ ]:
import torch
n_gpus = torch.cuda.device_count()

if n_gpus >= 2:
    print(f"🚀 Launching DDP on {n_gpus} GPUs via torchrun...")
    !torchrun --nproc_per_node=2 train.py \
        --data tusgan_encode.npz \
        --epochs 250 \
        --batch 512 \
        --seed 42 \
        --checkpoint_dir checkpoints \
        --log_dir runs/tusgan_v4_1_hard
else:
    print("🚀 Launching training on 1 GPU...")
    !python train.py \
        --data tusgan_encode.npz \
        --epochs 250 \
        --batch 512 \
        --seed 42 \
        --checkpoint_dir checkpoints \
        --log_dir runs/tusgan_v4_1_hard

### Step 5: Run Ultimate Evaluation on Final Checkpoint
Evaluates $N = 30,000$ synthetic schedules against ITUS 2019 microdata across all 5 evaluation domains (JSD, Time-Slice Transition F-Norm, Child Labor Violation, Spell-Duration EMD, Adversarial RF AUC).

In [ ]:
!python evaluate_ultimate.py \
    --checkpoint checkpoints/final.pt \
    --data tusgan_encode.npz \
    --n_samples 30000 \
    --output_dir evaluation_results

### Step 6: Review Table 6 Benchmark Summary

In [ ]:
import pandas as pd
import json

csv_path = "evaluation_results/table6_cell_c_summary.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)

json_path = "evaluation_results/metrics_cell_c.json"
if os.path.exists(json_path):
    with open(json_path) as f:
        metrics = json.load(f)
    print("\nDetailed Results Summary:")
    print(f"  • Macro JSD                    : {metrics[macro_jsd]:.6f}")
    print(f"  • Transition Matrix F-Norm     : {metrics[transition_fnorm]:.4f}")
    print(f"  • Child Labor Raw Violation    : {metrics[child_labor_raw_rate]:.2f}%")
    print(f"  • Mean Spell Duration EMD      : {metrics[mean_spell_emd_min]:.2f} min ({metrics[mean_spell_emd_slots]:.3f} slots)")
    print(f"  • Adversarial Random Forest AUC: {metrics[adversarial_auc]:.4f}")

### Step 7: Display Diagnostic Publication Plots

In [ ]:
from IPython.display import Image, display

diurnal_plot = "evaluation_results/cell_c_diurnal_participation.png"
if os.path.exists(diurnal_plot):
    print("Diurnal Participation Curves (Real vs Synthetic Cell C):")
    display(Image(diurnal_plot, width=900))

roc_plot = "evaluation_results/cell_c_adversarial_roc.png"
if os.path.exists(roc_plot):
    print("Adversarial Validation ROC Curve:")
    display(Image(roc_plot, width=500))

### Step 8: Package Artifacts for Direct Download
Bundles the trained weights (`final.pt`), gradient diagnostics, evaluation metrics, and TensorBoard logs into a single zip file in `/kaggle/working/`.

In [ ]:
!zip -r /kaggle/working/tusgan_v4_1_trained_outputs.zip checkpoints/final.pt checkpoints/grad_diagnostics_*.pt evaluation_results/ runs/
print("\n🎉 Finished! Download tusgan_v4_1_trained_outputs.zip from the Kaggle Output pane on the right.")